In [1]:
from datetime import date
import netCDF4 as nc
import os
import pandas as pd
import sys
import time
import fiona
import xarray as xr

In [2]:
sys.path.append('D:\\repos\\E-OBS-SWB2\\Python')

In [3]:
from EOBSobject import EOBSobject
from RechargeCalc import RechargeCalc

In [5]:
cwd = 'D:/Dati pesanti/SWB2_MAURICE'
cwd

'D:/Dati pesanti/SWB2_MAURICE'

## Use EOBSobject

In [6]:
outpath = os.path.join(cwd, 'climate_ncfile')
inpath = os.path.join(cwd, 'data_original', 'E-OBS')

In [7]:
vars = ['rr', 'tn', 'tx']
outnames = ['prcp', 'tmin', 'tmax']

coord = {'lon': [8.691, 8.929, 9.524, 9.537],
            'lat': [45.611, 45.308, 45.610, 45.306]}
coord = pd.DataFrame(coord)
# need one file per year
start = 2019
end = 2024

In [9]:
for i, var in enumerate(vars):
    f = EOBSobject(var, inpath, outpath, folder = True, swb2 = True)
    f.load()
    # cut in space and time
    f.set_outname(outnames[i])
    f.cut_spacetime(coord, start, end, option = 'singleyear', contourcell=2, autosave = True, readme = True)
    f.close_netcdf()

In [77]:
# load swb2 output and cut it
# from june 2023 to september 2024
# and in the MAURICE area
swb2path = os.path.join('\\'.join(cwd.split('\\')[:-1]), 'output', 'ModelMI_net_infiltration__2019-01-01_2024-12-31__338_by_660.nc')
outswb2 = xr.open_dataset(swb2path, engine='netcdf4')

In [78]:
outswb2

<xarray.Dataset> Size: 2GB
Dimensions:           (time: 2192, y: 338, x: 660)
Coordinates:
  * time              (time) datetime64[ns] 18kB 2019-01-01 ... 2024-12-31
  * y                 (y) float64 3kB 5.051e+06 5.051e+06 ... 5.017e+06
  * x                 (x) float64 5kB 1.476e+06 1.476e+06 ... 1.542e+06
    lat               (y, x) float64 2MB ...
    lon               (y, x) float64 2MB ...
Data variables:
    net_infiltration  (time, y, x) float32 2GB ...
    crs               int32 4B ...
Attributes:
    source:              net_infiltration output from SWB run started on May ...
    executable_version:  version 2.0 Beta, Git branch:  master, Git commit ha...
    conventions:         CF-1.6
    history:             May 14 2025 16:35:16: Soil-Water-Balance run started.

In [79]:
from datetime import datetime

In [80]:
datetime(2019,1,1) - datetime(2023, 6, 4)

datetime.timedelta(days=-1615)

In [81]:
datetime(2019,1,1) - datetime(2024, 9, 29)

datetime.timedelta(days=-2098)

In [82]:
cut = outswb2.isel(time=slice(1615, 2098+1)).copy()

In [83]:
cut.to_netcdf(os.path.join('\\'.join(cwd.split('\\')[:-1]), 'output', 'ModelMI_net_infiltration__2023-06-04_2024-09-29__338_by_660.nc'))

In [84]:
check = xr.load_dataset(os.path.join('\\'.join(cwd.split('\\')[:-1]), 'output', 'ModelMI_net_infiltration__2023-06-04_2024-09-29__338_by_660.nc'), format = 'netcdf4')

## Use RechargeCalc

In [ ]:
start = time.time()

cell_area = 100*100 #m2
#Path to the SWB2 output
swb2path = os.path.join('\\'.join(cwd.split('\\')[:-1]), 'output', "ModelMI_net_infiltration__2023-06-04_2024-09-29__338_by_660.nc")
#Path to the input .csv files folder
inputpath = os.path.join('\\'.join(cwd.split('\\')[:-1]),'data_original', 'file_input_rechargecalc', 'swb_maurice')
sppath = os.path.join(inputpath, 'rirrigua_speciale_swb_maurice.csv')

r = RechargeCalc(cell_area, uniqueid = 'indicatore', nSP = 69)
r.load_inputfiles(swb2path, inputpath)

frequency = 7

r.meteoricR(frequency = frequency, units = 'ms', fixrow=1, fixcol=4)

coeffs = {
    'E': 0.3,  #Irrigation technique efficiency
    'R': 0.05, #Residual runoff
    'RISP': 1, #1 - fraction of water saved by a change of irrigation technique
    'P': 1     #Percentage of the cell covered by the irrigation
    }

col = ['land_cover', 'land_cover', 'zona_urbana']
valcol = [123, 124, 1]
option = [0, 1] #0: OR, 1: AND

r.urbanR(coeff=0.125, col=col, valcol=valcol, option=option)

r.irrigationR(coeffs, specialpath=sppath)

r.totalR(fillna=True)

r.export('recharge','rtot',
             outpath = os.path.join('\\'.join(cwd.split('\\')[:-1]), 'rtot'),
             outname = 'rtot_swb_maurice',
             withcoord=True,
             coordpath = os.path.join(inputpath, 'coord.csv'))
r.georef('recharge','rtot',
            outpath = os.path.join('\\'.join(cwd.split('\\')[:-1]), 'rtot'),
            fname = 'rtot_swb_maurice.shp', 
            coordpath = os.path.join(inputpath, 'coord.csv'),
            crs = 'epsg:3003', dropcoord=False, driver = 'ESRI Shapefile')
end = time.time()
print((end - start)/60, 'min')

Loading the input files
-----------------------
indicatori file found
ricarica_irrigua file found
extractions file found
Meteoric recharge dataframe creation
------------------------------------
Performing the sum of net_infiltration over the stress periods provided
Output unit measure: ms
End of the procedure
Elapsed time: 8.81 s
Urban recharge dataframe creation
---------------------------------
Elapsed time: 20.13 s
Irrigation recharge dataframe creation
--------------------------------------
Elapsed time: 2.41 s
Total recharge dataframe creation
---------------------------------
Elapsed time: 1.69 s
18.392095804214478 s
Shapefile saved in d:\Dati pesanti\SWB2_MAURICE\rtot as rtot_swb_maurice.shp
Elapsed time: 105.48 s
2.639388672510783 min
